In [100]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv
/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/data_description.txt
/kaggle/input/competitions/home-data-for-ml-course/test.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv
/kaggle/input/competitions/home-data-for-ml-course/test.csv


# Introduction to Machine Learning

## Model Validation and Random Forest

This notebook covers model validation using Decision Tree and Random Forest models to predict house prices using the Iowa Housing dataset.

## 1. Preparing the Data

In [101]:
# Importing required libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Loading the dataset
housing_file_path = "/kaggle/input/competitions/home-data-for-ml-course/"

housing_test_data = pd.read_csv(housing_file_path + "test.csv")
housing_train_data = pd.read_csv(housing_file_path + "train.csv")


# Exploring the training dataset
print("Train data:")
print(housing_train_data.head())

# Checking dataset dimensions
print("Original dataset:", housing_train_data.shape)

# Displaying the last five rows
print("Last five rows:")
print(housing_train_data.tail())

# Checking column names, data types and non-null values
print("Dataset information:")
housing_train_data.info()

# Viewing statistical summary of numerical columns
print("\nStatistical summary:")
print(housing_train_data.describe())

# Checking for duplicate rows
print("\nDuplicate rows:")
print(housing_train_data.duplicated().sum())

# Displaying all column names
print("\nColumn names:")
print(housing_train_data.columns)

# Checking missing values in each column
print("\nMissing values:")
print(housing_train_data.isnull().sum())


Train data:
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008 

## 2. Data Cleansing

In [102]:

# Data Cleaning and Handling Missing Values

# Checking columns with missing values
missing_values = housing_train_data.isnull().sum()
print(missing_values[missing_values > 0].sort_values(ascending=False))

# Identifying numerical and categorical columns
print("Numerical columns:")
print(housing_train_data.select_dtypes(include=['number']).columns)

print("\nCategorical columns:")
print(housing_train_data.select_dtypes(include=['object']).columns)

# Separating numerical and categorical columns
numerical_columns = housing_train_data.select_dtypes(include=['number']).columns
categorical_columns = housing_train_data.select_dtypes(include=['object']).columns

# Checking missing values in numerical columns
print("Missing numerical values:")
print(housing_train_data[numerical_columns].isnull().sum().sort_values(ascending=False))

# Checking missing values in categorical columns
print("\nMissing categorical values:")
print(housing_train_data[categorical_columns].isnull().sum().sort_values(ascending=False))


# Handling missing values in numerical columns using median
numerical_columns = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea']

for column in numerical_columns:
    housing_train_data[column] = housing_train_data[column].fillna(
        housing_train_data[column].median()
    )

# Verifying numerical columns after cleaning
print(housing_train_data[numerical_columns].isnull().sum())


# Handling missing values in categorical columns

# Group 1: Missing values indicate that the feature does not exist
categorical_none_columns = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence',
    'MasVnrType', 'FireplaceQu',
    'GarageFinish', 'GarageQual', 'GarageType', 'GarageCond',
    'BsmtFinType2', 'BsmtExposure', 'BsmtFinType1',
    'BsmtQual', 'BsmtCond'
]

for column in categorical_none_columns:
    housing_train_data[column] = housing_train_data[column].fillna('None')


# Group 2: Missing information is replaced with the most frequent value (mode)
housing_train_data['Electrical'] = housing_train_data['Electrical'].fillna(
    housing_train_data['Electrical'].mode()[0]
)

# Final check for remaining missing values
print(housing_train_data.isnull().sum()[housing_train_data.isnull().sum() > 0])


PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64
Numerical columns:
Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
       'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
       'MiscVal', 'MoSold', 'YrSold', 'SalePr

## 3. Decision Tree Regressor and Random Forest Regressor with model performance comparison

In [103]:

# Machine Learning Models

# Separating features (X) and target (y)
X = housing_train_data.drop('SalePrice', axis=1)

# Selecting numerical features for model training
X = X.select_dtypes(exclude=['object'])

y = housing_train_data['SalePrice']

# Checking the shape of features and target
print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Splitting the data into training and validation sets
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)


# 1. Decision Tree Regressor

iowa_model = DecisionTreeRegressor(random_state=1)

# Training the Decision Tree model
iowa_model.fit(train_X, train_y)

# Making predictions on validation data
val_predictions = iowa_model.predict(val_X)

# Calculating Mean Absolute Error (MAE)
val_mae = mean_absolute_error(val_y, val_predictions)

print("Validation MAE for Decision Tree Regressor (default): {:,.0f}".format(val_mae))


# 2. Improving the Decision Tree

iowa_model = DecisionTreeRegressor(max_leaf_nodes=100, random_state=1)

# Training the Decision Tree with limited leaf nodes
iowa_model.fit(train_X, train_y)

# Making predictions on validation data
val_predictions = iowa_model.predict(val_X)

# Calculating MAE
val_mae = mean_absolute_error(val_y, val_predictions)

print("Validation MAE for Decision Tree Regressor (max_leaf_nodes=100): {:,.0f}".format(val_mae))


# 3. Random Forest Regressor

rf_model = RandomForestRegressor(random_state=1)

# Training the Random Forest model
rf_model.fit(train_X, train_y)

# Making predictions on validation data
rf_val_predictions = rf_model.predict(val_X)

# Calculating MAE
rf_val_mae = mean_absolute_error(val_y, rf_val_predictions)

print("Validation MAE for Random Forest Regressor (default): {:,.0f}".format(rf_val_mae))


# 4. Improving the Random Forest Regressor

rf_model = RandomForestRegressor(max_leaf_nodes=100, random_state=1)

# Training the Random Forest with limited leaf nodes
rf_model.fit(train_X, train_y)

# Making predictions on validation data
rf_val_predictions = rf_model.predict(val_X)

# Calculating MAE
rf_val_mae = mean_absolute_error(val_y, rf_val_predictions)

print("Validation MAE for Random Forest Regressor (max_leaf_nodes=100): {:,.0f}".format(rf_val_mae))


Features shape: (1460, 37)
Target shape: (1460,)
Validation MAE for Decision Tree Regressor (default): 26,741
Validation MAE for Decision Tree Regressor (max_leaf_nodes=100): 24,931
Validation MAE for Random Forest Regressor (default): 16,925
Validation MAE for Random Forest Regressor (max_leaf_nodes=100): 17,541


## Random Forest submission

In [105]:
# Preparing test data
test_X = housing_test_data.select_dtypes(exclude=['object'])
# Random Forest predictions
rf_test_preds = rf_model.predict(test_X)

rf_output = pd.DataFrame({
    'Id': housing_test_data.Id,
    'SalePrice': rf_test_preds
})

rf_output.to_csv('submission.csv', index=False)

## Conclusion

In this task, Decision Tree and Random Forest regression models were trained and evaluated using the Iowa Housing dataset.

The models were evaluated using Mean Absolute Error (MAE) to measure the difference between predicted and actual house prices.

The Decision Tree model was improved by setting max_leaf_nodes=100, reducing its validation MAE from 26,741 to 24,931.

The Random Forest model achieved better results, with a validation MAE of 16,925 without specifying max_leaf_nodes. Setting max_leaf_nodes=100 increased its MAE to 17,541.

Overall, the default Random Forest model performed best among the four tested models.